# Neural ODE Architecture Sweep — Kaggle Runner

Runs the Euler and/or Heat neural ODE architecture sweep scripts.

**Expected runtime**: ~1-1.5 hours per experiment on T4 GPU

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURE WHICH EXPERIMENTS TO RUN                        ║
# ╚══════════════════════════════════════════════════════════════╝
RUN_EULER = True
RUN_HEAT  = True


In [ ]:
!pip install -q --upgrade "jax[cuda12]" equinox diffrax optax opinf numpyro

In [ ]:
import jax
devices = jax.devices()
print(f"JAX {jax.__version__} — devices: {devices}")
assert any(d.platform == "gpu" for d in devices), "No GPU! Enable in Settings > Accelerator"

In [ ]:
%cd /kaggle/working
!rm -rf probabilistic_rom_inference
!git clone --branch tests https://github.com/Anthony50102/probabilistic_rom_inference.git
%cd probabilistic_rom_inference

In [ ]:
import os, subprocess, zipfile, glob

REPO = "/kaggle/working/probabilistic_rom_inference"
SWEEP = os.path.join(REPO, "experiments", "neural_ode")
FIGS = os.path.join(SWEEP, "figures")
os.makedirs(FIGS, exist_ok=True)

print("Sweep dir:", SWEEP)
print("Files:", os.listdir(SWEEP))

In [ ]:
if RUN_EULER:
    print("=" * 70)
    print("Running Euler architecture sweep...")
    print("=" * 70)
    r = subprocess.run(
        ["python", "-u", "euler_neural_ode.py"],
        cwd=SWEEP,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    print(f"Euler exit code: {r.returncode}")
else:
    print("Skipping Euler (RUN_EULER=False)")

In [ ]:
if RUN_HEAT:
    print("=" * 70)
    print("Running Heat architecture sweep...")
    print("=" * 70)
    r = subprocess.run(
        ["python", "-u", "heat_neural_ode.py"],
        cwd=SWEEP,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    print(f"Heat exit code: {r.returncode}")
else:
    print("Skipping Heat (RUN_HEAT=False)")

In [ ]:
ZIP_PATH = "/kaggle/working/neural_ode_sweep_results.zip"

# Collect ALL files under figures/ (any depth)
all_files = []
for root, dirs, files in os.walk(FIGS):
    for f in files:
        all_files.append(os.path.join(root, f))

print(f"Found {len(all_files)} files to zip:")
for fp in sorted(all_files):
    sz = os.path.getsize(fp)
    print(f"  {os.path.relpath(fp, SWEEP):50s}  {sz/1024:.1f} KB")

if len(all_files) == 0:
    print("WARNING: No figures found! Check for errors above.")
else:
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
        for fp in all_files:
            arcname = os.path.relpath(fp, SWEEP)
            zf.write(fp, arcname)
    # Verify the zip
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        names = zf.namelist()
        bad = zf.testzip()
    print(f"\nZipped {len(names)} files to {ZIP_PATH}")
    if bad:
        print(f"ERROR: corrupt file in zip: {bad}")
    else:
        print("Zip integrity check: OK")
    print("Download from the Output tab \u2192")